# Stage 1 — Proper Evaluation with Held-Out Test Set
Uses 80/10/10 split. Reports real metrics, not 8 anecdotal examples.

In [ ]:
!pip install -q transformers torch scikit-learn tqdm

In [ ]:
import json
from collections import Counter

import torch
from google.colab import files
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("Upload data/stage1_splits.json:")
uploaded = files.upload()

with open("stage1_splits.json") as f:
    splits = json.load(f)

X_train = [d["text"] for d in splits["train"]]
y_train = [d["label"] for d in splits["train"]]
X_val = [d["text"] for d in splits["val"]]
y_val = [d["label"] for d in splits["val"]]
X_test = [d["text"] for d in splits["test"]]
y_test = [d["label"] for d in splits["test"]]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Test dist: {dict(Counter(y_test))}")

In [ ]:
class ArgDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "label": torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_ds = ArgDataset(X_train, y_train, tokenizer)
val_ds = ArgDataset(X_val, y_val, tokenizer)
test_ds = ArgDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

best_val_f1 = 0
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/3"):
        out = model(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device), labels=batch["label"].to(device))
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += out.loss.item()

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            val_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            val_true.extend(batch["label"].numpy())
    val_f1 = f1_score(val_true, val_preds, average="binary")
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_f1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained("stage1_logical_detector")
        tokenizer.save_pretrained("stage1_logical_detector")

In [ ]:
# Final test evaluation
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("stage1_logical_detector").to(device)
model.eval()

test_preds, test_true = [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        test_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        test_true.extend(batch["label"].numpy())

print("="*60)
print(f"STAGE 1 — HELD-OUT TEST SET ({len(test_true)} samples)")
print("="*60)
print(classification_report(test_true, test_preds, target_names=["Non-Argument", "Argument"]))
print(f"Accuracy: {accuracy_score(test_true, test_preds):.4f}")
print(f"Binary F1: {f1_score(test_true, test_preds, average='binary'):.4f}")

In [ ]:
!zip -r stage1_logical_detector.zip stage1_logical_detector/
from google.colab import files

files.download("stage1_logical_detector.zip")